# Hexaly QPLIB Benchmarking

### Import libraries

In [5]:
import time
import numpy as np
import hexaly.optimizer
import os

### Set up Hexaly solver

In [6]:
def hexaly_solver(Q,c,sum_constraint, time_limit= 1800):
    n = len(np.squeeze(c))
    def external_xTQx(x):
        return x @ Q @ x

    def external_2Qx(x):
        return 2 * Q @ x
    
    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        model = optimizer.model
        x = [model.float(0.0,100.0) for _ in range(n)]
        linear_term = model.sum(c[i]*x[i] for i in range(n))
        xTQx_func = model.create_double_external_function(external_xTQx, external_2Qx)
        objective = linear_term + xTQx_func(x)

        model.constraint(model.sum(x) == sum_constraint)
        model.minimize(objective)
        model.close()

        optimizer.param.time_limit = time_limit
        print("Starting Hexaly solver...")
        optimizer.solve()
        solution = [x[i].value for i in range(n)]
        objective_val = objective.value
        print(f"Status: {optimizer.solution.status}")
        return solution, objective_val


### load data file

In [7]:

def read_qplib(path):
    with open(path) as fh:
        toks = [t for t in (line.split("#")[0].strip() for line in fh) if t]

    pos = 0

    def nxt():
        nonlocal pos
        val = toks[pos]
        pos += 1
        return val

    def defaulted(length):
        """Read a `default value` / `number of non-defaults` / entries block."""
        arr = np.full(length, float(nxt()))
        for _ in range(int(nxt())):
            i, v = nxt().split()
            arr[int(i) - 1] = float(v)
        return arr

    name = nxt()
    probtype = nxt()
    sense = nxt().lower()
    n = int(nxt())
    m = int(nxt())

    # quadratic terms of the objective (each unordered pair listed once)
    Q = np.zeros((n, n), dtype=np.float32)
    for _ in range(int(nxt())):
        i, j, v = nxt().split()
        i, j, v = int(i) - 1, int(j) - 1, float(v)
        if i == j:
            Q[i, i] = v
        else:
            Q[i, j] = Q[j, i] = 0.5 * v   # split the pair coefficient over both triangles

    b = defaulted(n)            # linear terms of the objective
    obj_const = float(nxt())

    # linear terms of the constraints
    A = np.zeros((m, n))
    for _ in range(int(nxt())):
        k, i, v = nxt().split()
        A[int(k) - 1, int(i) - 1] = float(v)

    inf = float(nxt())
    lhs, rhs = defaulted(m), defaulted(m)
    lb, ub = defaulted(n), defaulted(n)

    return dict(name=name, type=probtype, sense=sense, n=n, m=m,
                Q=Q, b=b, obj_const=obj_const,
                A=A, lhs=lhs, rhs=rhs, lb=lb, ub=ub, inf=inf)


### Solve with hexaly

In [8]:
# Instance details
inst_dir = "Instances/"
inst_name = "QPLIB_2761.qplib"
inst = read_qplib(os.path.join(inst_dir, inst_name))
n, Q, b = inst["n"], inst["Q"], inst["b"]
M = 0.5 * Q
c = b.copy()
sum_constraint = float(inst["rhs"][0])   # sum_i x_i = 1

In [9]:
# solver parameters
time_limit = 300

In [10]:
# solve with Hexaly
hexaly_start = time.time()
hex_sol_list, hex_obj = hexaly_solver(Q=M,c=c,sum_constraint=sum_constraint,time_limit=time_limit)
hexaly_end = time.time()
hex_sol = np.array(hex_sol_list)
hex_indices = np.where(hex_sol>1e-6)[0]

print(f"Eenergy value obtained by Hexaly:{hex_obj}")


Preprocess model 100%Starting Hexaly solver...
Push initial solution 100%
Model:  expressions = 1009, decisions = 500, constraints = 1, objectives = 1
Param:  time limit = 300 sec, no iteration limit

[objective direction ]:     minimize

[  0 sec,       0 itr]: No feasible solution found (infeas = 1)
[  1 sec,     226 itr]:     0.481681
[  2 sec,    1417 itr]:     0.415455
[  3 sec,    1417 itr]:     0.415455
[  4 sec,    2207 itr]:    0.0669736
[  5 sec,    2713 itr]:    0.0669736
[  6 sec,    3123 itr]:   0.00109363
[  7 sec,    3513 itr]:   0.00109363
[  8 sec,    4202 itr]:   0.00109363
[  9 sec,    4202 itr]:   0.00109363
[ 10 sec,    4521 itr]:   0.00109363
[ optimality gap     ]:      100.00%
[ 11 sec,    5075 itr]:   0.00109363
[ 12 sec,    5357 itr]:   0.00109363
[ 13 sec,    5357 itr]:   0.00109363
[ 14 sec,    5608 itr]:   0.00109363
[ 15 sec,    5866 itr]:   0.00109363
[ 16 sec,    6135 itr]:   0.00109363
[ 17 sec,    6641 itr]:   0.00109363
[ 18 sec,    6911 itr]:   0.001